## 一、探索性数据分析（EDA）

这一步的目的是搞清楚每列是什么意思、有没有缺失值、哪些特征影响生还。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

print(train.shape)   # (891, 12)
print(train.head())

### 查看缺失值

In [ ]:
train.isnull().sum()
# Age      177   ← 需要填充
# Cabin    687   ← 太多了，直接删除
# Embarked   2   ← 填充众数

### 探索特征与生还率的关系

In [ ]:
# 性别与生还率
print(train.groupby('Sex')['Survived'].mean())
# female    0.742
# male      0.189  ← 女性生还率远高于男性

# 船舱等级与生还率
print(train.groupby('Pclass')['Survived'].mean())
# 1    0.630
# 2    0.473
# 3    0.242  ← 等级越高生还率越高

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
train.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0], title='性别 vs 生还率')
train.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[1], title='舱位等级 vs 生还率')
train['Age'].hist(bins=30, ax=axes[2])
axes[2].set_title('年龄分布')
plt.tight_layout()
plt.show()

## 二、数据清洗

In [ ]:
# 合并 train 和 test 一起处理，保证一致性
all_data = pd.concat([train, test], axis=0, ignore_index=True)

# 1. 填充 Age：用中位数填充
all_data['Age'] = all_data['Age'].fillna(all_data['Age'].median())

# 2. 填充 Embarked：用众数填充（S 出现最多）
all_data['Embarked'] = all_data['Embarked'].fillna(all_data['Embarked'].mode()[0])

# 3. 填充 Fare：test 里有 1 个缺失
all_data['Fare'] = all_data['Fare'].fillna(all_data['Fare'].median())

# 4. 删除无用列
all_data.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], inplace=True)

# 验证
print(all_data.isnull().sum())


## 三、特征工程

这是提升成绩的关键步骤——从已有数据中创造更有价值的特征

In [ ]:
# 1. 性别编码（文字 → 数字）
all_data['Sex'] = all_data['Sex'].map({'male': 0, 'female': 1})

# 2. Embarked 编码
all_data['Embarked'] = all_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 3. 创建新特征：家庭总人数
#    SibSp = 兄弟姐妹+配偶，Parch = 父母+孩子，+1 是本人
all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1

# 4. 创建新特征：是否独自一人
all_data['IsAlone'] = (all_data['FamilySize'] == 1).astype(int)

# 5. 年龄分组（把连续变量变成类别）
all_data['AgeBand'] = pd.cut(all_data['Age'],
                              bins=[0, 12, 18, 35, 60, 100],
                              labels=[0, 1, 2, 3, 4])
all_data['AgeBand'] = all_data['AgeBand'].astype(int)

# 查看结果
print(all_data.head())

## 四、模型训练

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# 分离回训练集和测试集
train_clean = all_data[all_data['Survived'].notna()]
test_clean  = all_data[all_data['Survived'].isna()]

# 选择特征列
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked',
            'FamilySize', 'IsAlone', 'AgeBand']

X = train_clean[features]
y = train_clean['Survived'].astype(int)
X_test = test_clean[features]

# 标准化（对逻辑回归重要）
scaler = StandardScaler()
X_scaled      = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# 模型1：逻辑回归（基线，最简单）
lr = LogisticRegression(max_iter=1000)
scores_lr = cross_val_score(lr, X_scaled, y, cv=5, scoring='accuracy')
print(f"逻辑回归 交叉验证准确率: {scores_lr.mean():.4f} ± {scores_lr.std():.4f}")

# 模型2：随机森林（通常更好）
rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores_rf = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f"随机森林 交叉验证准确率: {scores_rf.mean():.4f} ± {scores_rf.std():.4f}")

## 五、调参和优化

In [ ]:
from sklearn.model_selection import GridSearchCV

# 用网格搜索找最优超参数
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1  # 用所有 CPU 核并行
)
grid_search.fit(X, y)

print("最优参数:", grid_search.best_params_)
print("最优准确率:", grid_search.best_score_)

best_model = grid_search.best_estimator_

# 查看特征重要性（哪些特征对模型最有用）
importances = pd.Series(best_model.feature_importances_, index=features)
importances.sort_values().plot(kind='barh', title='特征重要性')
plt.show()

## 六、生成预测结果并提交

In [ ]:
# 训练最终模型
best_model.fit(X, y)

# 对测试集进行预测
predictions = best_model.predict(X_test)

# 读回原始 test.csv 获取 PassengerId
test_raw = pd.read_csv('data/test.csv')

# 生成提交文件
submission = pd.DataFrame({
    'PassengerId': test_raw['PassengerId'],
    'Survived':    predictions
})
# submission.to_csv('submission.csv', index=False)

print(submission.head(10))
print(f"预测生还人数: {predictions.sum()}")